Run the updated BindCraft installation and DWV VP3 campaign setup.

This is the same campaign that produced **`l79_s469532_mpnn9`**, the one DWV VP3
design that actually bound (KD 546 / 666 nM). Target prep and hotspots are
preserved verbatim from that run. What changed is the **install**, which is what
broke last time.

### What was wrong in v1
| Bug | Consequence | Fix |
|---|---|---|
| `touch params/done.txt` ran **before** the pip installs | one failed pip poisoned the cache forever — every later run saw `done.txt` and skipped the installs, leaving a permanently broken tree | marker is written **only** after the params tarball extracts; every component is checked independently |
| single all-or-nothing install block | any failure meant re-downloading 4 GB of AF2 weights | each component installs and repairs on its own |
| `assert d.platform == "gpu"` | newer JAX reports `cuda`, so the guard fired on a perfectly good GPU | accepts `gpu` and `cuda`, and checks `jax.default_backend()` |
| no PyRosetta fallback | the RosettaCommons `--find-links` host is flaky; when it blipped, install died | falls back to the official `pyrosetta-installer` |

### How to run
Runtime ▸ Change runtime type ▸ **GPU** (T4 is fine), then Run All.
Everything lives on Drive, so a disconnect costs you nothing — just Run All again.

## 0 · Drive guard — run FIRST, every session

If Drive is not writable, stop now rather than three hours in.

In [ ]:
import os, sys, time, json, glob, subprocess
from google.colab import drive

MOUNT = "/content/drive"
if not os.path.isdir(f"{MOUNT}/MyDrive"):
    drive.mount(MOUNT)

probe = f"{MOUNT}/MyDrive/.bindcraft_write_probe"
try:
    stamp = str(time.time())
    open(probe, "w").write(stamp)
    assert open(probe).read() == stamp
    os.remove(probe)
    print("Drive is mounted and writable.")
except Exception as e:
    raise SystemExit(f"FATAL: Google Drive is not writable ({e}). "
                     "Re-mount it before continuing — do not proceed.")

## 1 · Config — the only cell you normally edit

In [ ]:
# ============================== CHOOSE TARGET ==============================
TARGET = "VP3"          # "VP3"  -> DWV VP3 P-domain (the campaign that worked)
                        # "deGFP" -> eGFP Minimizer face
GROUP  = "A_ridge"      # see TARGETS[...]["groups"] below
# ===========================================================================

TARGETS = {
    "VP3": {
        "pdb_id": "5G51", "chain": "A", "lo": 260, "hi": 416,
        "n_res": 157,
        # 5G51 is 1.45 A and full of alternate conformers. 21 residues have NO
        # blank-altloc copy, so a blank-only filter silently deletes them —
        # including 291-294 and 303, right in the surface of interest.
        "altloc_only": [263,270,271,278,291,292,293,294,303,313,318,328,
                        350,352,353,363,375,392,400,405,411],
        "gaps": [],
        "groups": {
            "A_ridge": "325,326,384,386",  # Q325 P326 V384 N386 — PRODUCED THE BINDER
            "B_triad": "277,294",          # opposite face, comparison arm ONLY.
    # B_triad is named after a PUTATIVE catalytic triad (Asp294-His277-Ser278) that did not
    # survive scrutiny: no hydrogen-bond geometry between any pair, and the only functional
    # assay ever run against it was negative. It is geometrically NON-CATALYTIC. Kept as a
    # Comparison arm; the functional assay was negative.
            "free":    "",                 # unconstrained; AF2 picks the site
        },
        "lengths": [60, 130],
    },
    "deGFP": {
        "pdb_id": "4EUL", "chain": "A", "lo": 6, "hi": 229,
        "n_res": 221,
        "altloc_only": [18,30,44,97,102,113,124,184,222],
        # 64->68 is the CRO chromophore, stored as HETATM; the break is expected.
        "gaps": [(64, 68)],
        "groups": {
            "M_core": "151,166,180,200",   # published Minimizer epitope core
            "M_wide": "149,151,164,166,178,180,182,198,200",
            "E_face": "221,223,224",       # Enhancer face — negative control arm
            "free":   "",
        },
        "lengths": [60, 110],
    },
}

T = TARGETS[TARGET]
HOTSPOTS = T["groups"][GROUP]
LENGTHS  = T["lengths"]
N_FINAL  = 20               # accepted-design goal; accumulates across sessions
FILTERS  = "default_filters.json"
ADVANCED = "default_4stage_multimer.json"
MAX_RESTARTS = 50

DRIVE_ROOT = "/content/drive/MyDrive/BindCraft"    # MyDrive — no space
RUN_NAME = f"{TARGET}_{GROUP}" + (f"_hs{HOTSPOTS.replace(',', '-')}" if HOTSPOTS else "_freebind")

# One run per folder, always. BindCraft appends to its CSVs with no locking —
# two runs sharing a design_path will corrupt each other's results.
DESIGN_PATH  = f"{DRIVE_ROOT}/{RUN_NAME}/"         # MUST end with "/"
SETTINGS_DIR = f"{DRIVE_ROOT}/{RUN_NAME}_settings"
LOG_DIR      = f"{DRIVE_ROOT}/{RUN_NAME}_logs"
for d in (DESIGN_PATH, SETTINGS_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

print(f"target   : {TARGET}  ({T['pdb_id']} chain {T['chain']} {T['lo']}-{T['hi']})")
print(f"arm      : {GROUP}   hotspots = {HOTSPOTS or '(free binding)'}")
print(f"lengths  : {LENGTHS}   goal = {N_FINAL} accepted")
print(f"drive    : {DESIGN_PATH}")
os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

## 2 · Install — idempotent and self-repairing

Each component is checked and installed on its own. Re-running this cell after a
partial failure repairs only what is missing; it never re-downloads the 4 GB of
AlphaFold weights you already have.

In [ ]:
WORK        = "/content/bindcraft"
PARAMS_DIR  = f"{WORK}/params"
PARAMS_DONE = f"{PARAMS_DIR}/done.txt"
AF2_TAR_URL = "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar"

def have(mod):
    try:
        __import__(mod); return True
    except Exception:
        return False

def sh(cmd):
    print(f"  $ {cmd[:110]}")
    return subprocess.call(cmd, shell=True)

# --- 1. BindCraft source ------------------------------------------------------
if not os.path.isdir(f"{WORK}/functions"):
    print("[1/4] cloning BindCraft")
    sh(f"git clone -q https://github.com/martinpacesa/BindCraft {WORK}")
else:
    print("[1/4] BindCraft source present")
for exe in ("dssp", "DAlphaBall.gcc"):
    p = f"{WORK}/functions/{exe}"
    if os.path.exists(p):
        os.chmod(p, 0o755)

# --- 2. AlphaFold2 params -----------------------------------------------------
# done.txt is written ONLY after the tar extracts. This is the v1 bug: the marker
# used to be written before the pip steps, so a failed pip left a "complete"
# install that could never self-heal.
if os.path.isfile(PARAMS_DONE):
    print("[2/4] AF2 params present")
else:
    print("[2/4] downloading AF2 params (~4 GB, several minutes)")
    os.makedirs(PARAMS_DIR, exist_ok=True)
    sh("apt-get install -y aria2 -qq")
    tar = "/content/af2.tar"
    if sh(f"aria2c -q -x 16 -s 16 -o af2.tar -d /content {AF2_TAR_URL}") != 0:
        sh(f"wget -q -O {tar} {AF2_TAR_URL}")
    if sh(f"tar -xf {tar} -C {PARAMS_DIR}") != 0:
        raise SystemExit("FATAL: AF2 params failed to extract. Re-run this cell.")
    os.remove(tar)
    open(PARAMS_DONE, "w").write("ok")      # <-- only now
    print("      params extracted")

# --- 3. ColabDesign -----------------------------------------------------------
if have("colabdesign"):
    print("[3/4] colabdesign present")
else:
    print("[3/4] installing ColabDesign")
    sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

# --- 4. PyRosetta (two routes; the RosettaCommons host is flaky) ---------------
if have("pyrosetta"):
    print("[4/4] pyrosetta present")
else:
    print("[4/4] installing PyRosetta (route 1: rosettacommons find-links)")
    sh("pip install -q pyrosetta --find-links "
       "https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization")
    if not have("pyrosetta"):
        print("      route 1 failed — falling back to pyrosetta-installer")
        sh("pip install -q pyrosetta-installer")
        sh('python -c "import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()"')

print("\ninstall pass complete — now run the preflight cell.")

## 3 · Preflight — fail loudly here, not three hours in

In [ ]:
problems = []

# GPU. Newer JAX reports the platform as "cuda"; v1 asserted == "gpu" and
# rejected perfectly good runtimes.
try:
    import jax
    devs = jax.devices()
    backend = jax.default_backend()
    print("jax", jax.__version__, "| backend:", backend, "| devices:", devs)
    if not (backend in ("gpu", "cuda") or any(d.platform in ("gpu", "cuda") for d in devs)):
        problems.append("No GPU visible to JAX. Runtime > Change runtime type > GPU.")
except Exception as e:
    problems.append(f"jax import failed: {e}")

for mod in ("colabdesign", "pyrosetta"):
    try:
        __import__(mod); print(f"{mod} OK")
    except Exception as e:
        problems.append(f"{mod} import failed: {e} — re-run the install cell.")

npz = glob.glob(f"{PARAMS_DIR}/params/*.npz") + glob.glob(f"{PARAMS_DIR}/*.npz")
size_gb = sum(os.path.getsize(f) for f in npz) / 1e9 if npz else 0
print(f"AF2 params: {len(npz)} files, {size_gb:.1f} GB")
if len(npz) < 5 or size_gb < 3.0:
    problems.append(f"AF2 params look incomplete ({len(npz)} files, {size_gb:.1f} GB). "
                    f"Delete {PARAMS_DONE} and re-run the install cell.")

for exe in ("dssp", "DAlphaBall.gcc"):
    p = f"{WORK}/functions/{exe}"
    if not (os.path.exists(p) and os.access(p, os.X_OK)):
        problems.append(f"{p} missing or not executable.")
    else:
        print(f"{exe} executable")

for f in (f"{WORK}/bindcraft.py", f"{WORK}/settings_filters/{FILTERS}",
          f"{WORK}/settings_advanced/{ADVANCED}"):
    if not os.path.exists(f):
        problems.append(f"missing {f}")

if problems:
    print("\n" + "="*70)
    for p in problems:
        print("PROBLEM:", p)
    raise SystemExit("Preflight failed — fix the above before running.")
print("\nPREFLIGHT PASSED — safe to run.")

## 4 · Prepare the target

Kept verbatim from the run that produced the binder. The altloc handling matters:
both structures have residues with **no blank-altloc copy**, and a naive
blank-only filter deletes them silently, right out of the surface you are targeting.

In [ ]:
import urllib.request

TARGET_PDB = f"{SETTINGS_DIR}/{T['pdb_id']}_chain{T['chain']}_{T['lo']}_{T['hi']}.pdb"

if not os.path.exists(TARGET_PDB):
    raw = f"/content/{T['pdb_id']}_raw.pdb"
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{T['pdb_id']}.pdb", raw)
    out, seen = [], set()
    for l in open(raw):
        if not l.startswith("ATOM"):        continue   # drops HETATM + waters
        if l[21] != T["chain"]:             continue
        try:    rn = int(l[22:26])
        except ValueError:                  continue
        if not (T["lo"] <= rn <= T["hi"]):  continue
        if l[16] not in (" ", "A"):         continue   # keep ONE conformer, not none
        key = (rn, l[12:16].strip())
        if key in seen:                     continue
        seen.add(key)
        out.append(l[:16] + " " + l[17:])              # blank the altloc column
    out.append("TER\nEND\n")
    open(TARGET_PDB, "w").writelines(out)

res  = sorted({int(l[22:26]) for l in open(TARGET_PDB) if l.startswith("ATOM")})
n_ca = sum(1 for l in open(TARGET_PDB) if l.startswith("ATOM") and l[12:16] == " CA ")
nat  = sum(1 for l in open(TARGET_PDB) if l.startswith("ATOM"))
gaps = [(a, b) for a, b in zip(res, res[1:]) if b != a + 1]
print(f"{TARGET_PDB}\n  {len(res)} residues {res[0]}-{res[-1]} | {n_ca} CA | {nat} atoms")
print(f"  chain breaks: {gaps or 'none'}")

# Tripwires — a silently wrong target is the most expensive failure mode here.
assert len(res) == T["n_res"],  f"expected {T['n_res']} residues, got {len(res)}"
assert n_ca    == T["n_res"],   f"expected {T['n_res']} CA, got {n_ca} (dedupe failed)"
assert gaps    == T["gaps"],    f"unexpected chain breaks {gaps}, expected {T['gaps']}"
missing = [r for r in T["altloc_only"] if r not in res]
assert not missing, f"altloc-only residues dropped: {missing} — the (' ','A') filter broke"
if HOTSPOTS:
    hs = [int(x) for x in HOTSPOTS.split(",")]
    absent = [h for h in hs if h not in res]
    assert not absent, f"hotspots {absent} are not in the prepared target"
    print(f"  all {len(hs)} hotspots present: {hs}")
print("  target OK")

## 5 · Write settings to Drive

In [ ]:
settings = {
    "design_path": DESIGN_PATH,
    "binder_name": RUN_NAME,
    "starting_pdb": TARGET_PDB,
    "chains": T["chain"],
    "target_hotspot_residues": HOTSPOTS,   # "" -> None internally -> AF2 picks the site
    "lengths": LENGTHS,
    "number_of_final_designs": N_FINAL,
}
SETTINGS_JSON = f"{SETTINGS_DIR}/target.json"
json.dump(settings, open(SETTINGS_JSON, "w"), indent=4)
print(json.dumps(settings, indent=4))
print("\nmode:", "FREE BINDING — no hotspot constraint" if not HOTSPOTS else f"hotspots {HOTSPOTS}")
print("hotspots are PDB AUTHOR numbers, never 0-indexed.")

## 6 · Run — supervised and resumable

Runs `bindcraft.py` as a subprocess and restarts it if it dies, which is what
makes this survive Colab's disconnects. Everything streams to a log on Drive, so
progress is never lost. Safe to interrupt and re-run at any point.

In [ ]:
import datetime
os.chdir(WORK)
log_path = f"{LOG_DIR}/run_{datetime.datetime.now():%Y%m%d_%H%M%S}.log"
print("log:", log_path)

def progress():
    return (len(glob.glob(DESIGN_PATH + "Accepted/*.pdb")),
            len(glob.glob(DESIGN_PATH + "Trajectory/Relaxed/*.pdb")))

for attempt in range(1, MAX_RESTARTS + 1):
    a, t = progress()
    if a >= N_FINAL:
        print(f"TARGET REACHED — {a}/{N_FINAL} accepted"); break
    print(f"\n===== attempt {attempt} | {a} accepted, {t} trajectories =====", flush=True)
    cmd = [sys.executable, "-u", "./bindcraft.py",
           "--settings",  SETTINGS_JSON,
           "--filters",  f"./settings_filters/{FILTERS}",
           "--advanced", f"./settings_advanced/{ADVANCED}"]
    with open(log_path, "a") as lf:
        lf.write(f"\n===== attempt {attempt} @ {datetime.datetime.now()} =====\n"); lf.flush()
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end=""); lf.write(line); lf.flush()   # durable on Drive
        except KeyboardInterrupt:
            p.terminate(); print("\nstopped by user"); break
        rc = p.wait()
    a2, t2 = progress()
    print(f"\nexit {rc} | accepted {a}->{a2} | trajectories {t}->{t2}")
    if a2 >= N_FINAL:
        print("TARGET REACHED"); break
    if a2 == a and t2 == t and rc != 0:
        print("WARNING: no progress this attempt. Check the log above before burning "
              "more GPU time — a repeating traceback will not fix itself.")
    time.sleep(20)

## 7 · Status — safe to run any time, from any session

In [ ]:
acc = sorted(glob.glob(DESIGN_PATH + "Accepted/*.pdb"))
traj = glob.glob(DESIGN_PATH + "Trajectory/Relaxed/*.pdb")
print(f"{RUN_NAME}\n  accepted    : {len(acc)}/{N_FINAL}")
print(f"  trajectories: {len(traj)}")
rate = (len(acc) / len(traj) * 100) if traj else 0
print(f"  accept rate : {rate:.1f}%")
for c in sorted(glob.glob(DESIGN_PATH + "*.csv")):
    try:
        import pandas as pd
        print(f"  {os.path.basename(c):<34} {len(pd.read_csv(c)):>5} rows")
    except Exception as e:
        print(f"  {os.path.basename(c):<34} unreadable ({e})")
logs = sorted(glob.glob(LOG_DIR + "/*.log"))
if logs:
    print(f"\nlatest log: {logs[-1]}")
    print("".join(open(logs[-1]).readlines()[-12:]))

## 8 · Export for `consensus_rank.py`

Writes a CSV of the accepted sequences with the metric columns the scorer expects.
Fill `protenix_iptm`, `protenix_seed_spread` and `esmfold_iptm` from your scoring
runs, then:

```
python scripts/consensus_rank.py bindcraft_designs.csv -o ranked.csv
```

In [ ]:
import csv as _csv
rows, seen = [], set()
for f in sorted(glob.glob(DESIGN_PATH + "Accepted/*.pdb")):
    name = os.path.basename(f)[:-4]
    seq, last = [], None
    aa3 = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
           "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
           "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"}
    for l in open(f):
        if l.startswith("ATOM") and l[21] == "B" and l[12:16] == " CA ":
            rn = l[22:26]
            if rn != last:
                seq.append(aa3.get(l[17:20].strip(), "X")); last = rn
    s = "".join(seq)
    if s and s not in seen:
        seen.add(s)
        rows.append({"name": name, "sequence": s, "length": len(s),
                     "protenix_iptm": "", "protenix_seed_spread": "",
                     "esmfold_iptm": "", "solublempnn_delta": ""})

out = f"{DRIVE_ROOT}/{RUN_NAME}_designs.csv"
if rows:
    with open(out, "w", newline="") as fh:
        w = _csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print(f"wrote {len(rows)} unique sequences -> {out}")
    for r in rows[:5]:
        print(f"  {r['name']:<34} {r['length']:>4} aa  {r['sequence'][:46]}...")
else:
    print("No accepted designs yet — run cell 6 first.")